In [1]:
import pandas as pd
import numpy as np
import os

# ==========================================
# KONFIGURASI
# ==========================================
INPUT_CSV = 'cleaned_transcript_mapping_eq_3_5.csv'
SEED = 42

print(f"Loading dataset utama dari: {INPUT_CSV}...\n")
df_full = pd.read_csv(INPUT_CSV)

# Dapatkan daftar semua subjek unik
subjects = df_full['subject'].unique()
print(f"Ditemukan {len(subjects)} subjek: {subjects}\n")

# List untuk menyimpan hasil OOV agar bisa dijadikan tabel
oov_summary_list = []

print("Mulai memproses split dan analisis OOV untuk setiap subjek...\n")

for subject in subjects:
    # 1. Filter dataset per subjek
    df_subject = df_full[df_full['subject'] == subject].copy()
    
    # 2. Ambil kalimat unik dan acak (seed tetap agar reprodusibel)
    np.random.seed(SEED)
    unique_sentences = df_subject['sentence'].unique()
    n_unique = len(unique_sentences)
    
    train_count = int(n_unique * 0.7)
    val_count = int(n_unique * 0.1)
    
    shuffled_sentences = np.random.permutation(unique_sentences)
    
    train_sentences = set(shuffled_sentences[:train_count])
    val_sentences = set(shuffled_sentences[train_count:train_count+val_count])
    test_sentences = set(shuffled_sentences[train_count+val_count:])
    
    # 3. Filter DataFrame berdasarkan split kalimat
    df_train = df_subject[df_subject['sentence'].isin(train_sentences)]
    df_val = df_subject[df_subject['sentence'].isin(val_sentences)]
    df_test = df_subject[df_subject['sentence'].isin(test_sentences)]
    
    # 4. Simpan ke CSV fisik
    train_file = f"{subject}_eq_3_5_train.csv"
    val_file = f"{subject}_eq_3_5_val.csv"
    test_file = f"{subject}_eq_3_5_test.csv"
    
    df_train.to_csv(train_file, index=False)
    df_val.to_csv(val_file, index=False)
    df_test.to_csv(test_file, index=False)
    
    # 5. Analisis OOV Karakter (Train vs Test)
    train_text = "".join(df_train['sentence'].tolist())
    test_text = "".join(df_test['sentence'].tolist())
    
    train_chars = set(train_text)
    test_chars = set(test_text)
    
    # Cari karakter yang ada di Test tapi tidak ada di Train
    oov_chars = test_chars - train_chars
    
    test_text_len = len(test_text)
    if test_text_len > 0:
        oov_occurrences = sum(1 for char in test_text if char in oov_chars)
        oov_rate = (oov_occurrences / test_text_len) * 100
    else:
        oov_occurrences = 0
        oov_rate = 0.0

    # 6. Simpan metrik ke dalam list untuk tabel
    oov_summary_list.append({
        'Subject': subject,
        'Train (Baris)': len(df_train),
        'Val (Baris)': len(df_val),
        'Test (Baris)': len(df_test),
        'Kamus Train': len(train_chars),
        'Kamus Test': len(test_chars),
        'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
        'Total Muncul OOV': oov_occurrences,
        'OOV Rate (%)': round(oov_rate, 4)
    })

print("✅ Selesai memproses semua file CSV!\n")

# ==========================================
# MENAMPILKAN TABEL SUMMARY OOV
# ==========================================
df_summary = pd.DataFrame(oov_summary_list)

# Menampilkan tabel (gunakan display agar format tabel Jupyter keluar)
print("--- RINGKASAN SPLIT & OOV KARAKTER PER SUBJEK ---")
display(df_summary)

Loading dataset utama dari: cleaned_transcript_mapping_eq_3_5.csv...

Ditemukan 11 subjek: <ArrowStringArray>
[ 'SUB2',  'SUB3',  'SUB4',  'SUB1',  'SUB7', 'SUB10', 'SUB12',  'SUB5',
 'SUB11',  'SUB6',  'SUB8']
Length: 11, dtype: str

Mulai memproses split dan analisis OOV untuk setiap subjek...

✅ Selesai memproses semua file CSV!

--- RINGKASAN SPLIT & OOV KARAKTER PER SUBJEK ---


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB2,35,5,10,23,23,z,1,0.3125
1,SUB3,35,5,10,23,21,-,0,0.0000
2,SUB4,70,10,20,24,22,-,0,0.0000
3,SUB1,70,10,20,24,23,-,0,0.0000
4,SUB7,62,9,19,23,23,-,0,0.0000
5,SUB10,25,3,8,23,21,-,0,0.0000
6,SUB12,0,0,1,0,10,acdgiknst,17,100.0000
7,SUB5,35,5,10,23,23,-,0,0.0000
8,SUB11,30,4,9,23,21,-,0,0.0000
9,SUB6,68,9,21,23,23,-,0,0.0000
